In [16]:
!pip install sentence-transformers chromadb groq pandas -q
print("installation completed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [17]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully!!")

All libraries imported successfully!!


In [21]:
GROQ_API_KEY="gsk_...L1Q0"
os.environ["gsk_...L1Q0"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized.")
print("Note:If you see an authentication error later,double-check your API key")

Groq API client initialized.
Note:If you see an authentication error later,double-check your API key


In [22]:
df=pd.read_csv('/content/college_notes.csv')
print("Shape of dataset:",df.shape)
print("\nColumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [23]:
df=pd.read_csv('college_notes.csv')
print("Shape of Data Set:",df.shape)
print("\nColumn Names:",df.columns.tolist())
print("\nFirst 3 Rows:")
print(df.head(3).to_string(index=False))

Shape of Data Set: (15, 4)

Column Names: ['note_id', 'subject', 'topic', 'content']

First 3 Rows:
note_id          subject         topic                                                                                                                                                                                                                  content
   N001 Data Engineering ETL Pipelines ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
   N002 Data Engineering SQL Databases        A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
   N003 Data Engineering Data Cleaning       Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted d

In [24]:
print("Subjects in the datset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (numbers of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subjects in the datset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python 

In [25]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared:{len(documents)}")
print(f"First document ID:{ids[0]}")
print(f"First metadata:{metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}...")

Total chunks prepared:15
First document ID:note_N001
First metadata:{'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [26]:
print("LOading embedding model...")
print("(This may take 30-60 seconds on first run-model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")
embedding_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Embedding model loaded successfully!")
test_embedding=embedding_model.encode("this is a test sentence.")
print(f"Test embedding shape:{test_embedding.shape}")
print(f"First 5 values of test embedding:{test_embedding[:5]}")

LOading embedding model...
(This may take 30-60 seconds on first run-model is being downloaded)
(Subsequent runs will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
Test embedding shape:(384,)
First 5 values of test embedding:[0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [27]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("Embedding model loaded successfully!")
print(f"Collection name:college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

Embedding model loaded successfully!
Collection name:college_notes_rag
Documents in collection so far:0


In [28]:
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape:{embeddings.shape}")
embeddings_list=embeddings.tolist()
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)
print(f"\n Documents successfully added to CHromaDB")
print(f"Total documents in collection:{collection.count()}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape:(15, 384)

 Documents successfully added to CHromaDB
Total documents in collection:15


In [29]:
def retrieve_relavant_chunks(question,top_k=3):
  """
  Given a user question,retrive a relavant document chunks fromChromaDB
  Parameters:
  question (str):The users question as a text string
  top_k(int):How top resuts to return (default:3)
  Returns:
    A dictionary containing retrieved documents,distances and metadata
  """
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results

In [32]:
test_question="what is ETL and how does it work in data engineering"
print(f"Test Question:{test_question}")
results=retrieve_relavant_chunks(test_question,top_k=3)
print("\nTop 3 Retrieved Chunks:")
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
   print(f"\nResult{i+1}:")
   print(f"Subject:{meta['subject']}")
   print(f"Topic:{meta['topic']}")
   print(f"Distance:{dist:.4f}")
   print(f"Content:{doc[:120]}...")

Test Question:what is ETL and how does it work in data engineering

Top 3 Retrieved Chunks:

Result1:
Subject:Data Engineering
Topic:ETL Pipelines
Distance:0.2041
Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result2:
Subject:Data Engineering
Topic:APIs and Data Collection
Distance:1.1100
Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result3:
Subject:Python Programming
Topic:Data Visualization
Distance:1.3892
Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


In [37]:
def build_context_from_results(results):
  """
  Format ChromaDB retrieval results into a context string.
  Parameters:
  results:The output from collection.query()-a dictionary
  Returns:
  context_str(str):A formatted string of all retreved document chunks
  """
  context_parts=[]
  for i,(doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
      )):
         chunk_text=f"[Source{i+1}:{meta['subject']} - {meta['topic']}]\n{doc}"
         context_parts.append(chunk_text)
  return "\n\n".join(context_parts)

In [4]:
def generate_rag_answer(question, context):
    """
    Send the retrieved context and question to the Groq LLM for answer generation.
    """
    system_prompt="""You are a helpful academic assistant for engineering students.
You will be given context retrieved from a college knowledge base and a student's question.
RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the information comes from when possible.
"""
    user_prompt = f"""Context from Knowledge Base:
{context}
Question:
{question}
"""
    response=groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )
    answer=response.choices[0].message.content
    return answer
print("RAG generation defined successfully!!")

RAG generation defined successfully!!


In [13]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Complete RAG pipeline: Given a question, retrieve relevant context and generate an answer.

    Parameters:
    question (str): The user's question
    top_k (int): Number of chunks to retrieve (default: 3)
    verbose (bool): Whether to print intermediate steps (default: True)

    Returns:
    answer (str): The final generated answer
    """

    if verbose:
        print(f"Question: {question}")

    # Step 1: Retrieve relevant chunks
    results = collection.query(
        query_texts=[question],
        n_results=top_k
    )

    if verbose:
        print(f"\nRetrieved {len(results['documents'][0])} chunks")

    # Step 2: Build context from retrieved chunks
    context = build_context_from_results(results)

    if verbose:
        print("\nContext Built Successfully")
        print("\nContext:")
        print(context)

    # Step 3: Generate answer using LLM
    answer = generate_rag_answer(question, context)

    if verbose:
        print("\nAnswer Generated Successfully")

    return answer


print("RAG pipeline defined successfully")

RAG pipeline defined successfully
